# OncoBind-ML — Decision Tree Track
## Model A → Model B → Model C

**Important:** `DDG.EXP` is a regression target. Real OncoBind tree splits use regression impurity (squared error / variance reduction). A separate Gini demo is included only for explaining classification-tree splitting.

## Common setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib

In [ ]:
from scipy.stats import pearsonr, spearmanr

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV, GridSearchCV, learning_curve, validation_curve

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend" / "app").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook inside the OncoBind-ML project folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEED = 42
TARGET = "DDG.EXP"
DATA_PATH = Path(os.environ.get("ONCOBIND_DATA_PATH", str(PROJECT_ROOT / "data" / "raw" / "MdrDB_Full_Scraped.csv")))
if not DATA_PATH.is_file() or DATA_PATH.stat().st_size == 0:
    raise FileNotFoundError(f"Upload the nonempty MdrDB dataset to {DATA_PATH}")
N_JOBS = int(os.environ.get("ONCOBIND_N_JOBS", "1"))
np.random.seed(SEED)


In [ ]:
df=pd.read_csv(DATA_PATH, low_memory=False)
df.shape

In [ ]:
LIG=[x for x in df.columns if x.startswith("LIG.")]
MUT=[x for x in df.columns if x.startswith("MUT.")]
ENV=[x for x in df.columns if x.startswith("ENV.")]
PLIP=[x for x in df.columns if x.startswith("PLIP.")]
VINA=[x for x in df.columns if x=="vina" or (x.startswith("vina") and x[4:].isdigit())]
P2=[x for x in df.columns if x.startswith(("P2.","P2dl.","P2dp."))]

In [ ]:
A_FEATURES=LIG+MUT+ENV
B_FEATURES=A_FEATURES+PLIP
C_FEATURES=B_FEATURES+VINA+P2
pd.Series({"A":len(A_FEATURES),"B":len(B_FEATURES),"C":len(C_FEATURES)})

In [ ]:
y=pd.to_numeric(df[TARGET],errors="coerce")
keep=y.notna()
df_clean=df.loc[keep].reset_index(drop=True)
y=y.loc[keep].reset_index(drop=True)

In [ ]:
idx=np.arange(len(df_clean))
dev_idx,test_idx=train_test_split(idx,test_size=.20,random_state=SEED)
y_dev=y.iloc[dev_idx].reset_index(drop=True)
y_test=y.iloc[test_idx].reset_index(drop=True)
len(dev_idx),len(test_idx)

In [ ]:
def metrics(y_true,pred):
    return {
        "R2":r2_score(y_true,pred),
        "RMSE":np.sqrt(mean_squared_error(y_true,pred)),
        "MAE":mean_absolute_error(y_true,pred),
        "Pearson":pearsonr(y_true,pred).statistic,
        "Spearman":spearmanr(y_true,pred).statistic
    }

In [ ]:
def actual_vs_pred(y_true,pred,title):
    plt.figure(figsize=(5.5,5.5))
    plt.scatter(y_true,pred,s=10,alpha=.3)
    lo=min(np.min(y_true),np.min(pred)); hi=max(np.max(y_true),np.max(pred))
    plt.plot([lo,hi],[lo,hi],"--")
    plt.xlabel("Actual DDG.EXP"); plt.ylabel("Predicted DDG.EXP"); plt.title(title)
    plt.tight_layout(); plt.show()

In [ ]:
def residual_plot(pred,residual,title):
    plt.figure(figsize=(6.5,4))
    plt.scatter(pred,residual,s=10,alpha=.3)
    plt.axhline(0,linestyle="--")
    plt.xlabel("Predicted DDG.EXP"); plt.ylabel("Residual"); plt.title(title)
    plt.tight_layout(); plt.show()

# Gini impurity — classification concept demo only

In [ ]:
gini_demo=pd.DataFrame({"interaction_score":[1,2,3,4,6,7,8,9],"class":[0,0,0,1,0,1,1,1]})
gini_demo

In [ ]:
def gini(labels):
    p=pd.Series(labels).value_counts(normalize=True)
    return 1-np.sum(p**2)
gini(gini_demo["class"])

In [ ]:
left=gini_demo[gini_demo["interaction_score"]<=5]
right=gini_demo[gini_demo["interaction_score"]>5]
g_left=gini(left["class"]); g_right=gini(right["class"])
g_left,g_right

In [ ]:
weighted=(len(left)/len(gini_demo))*g_left+(len(right)/len(gini_demo))*g_right
weighted

In [ ]:
gini_tree=DecisionTreeClassifier(max_depth=2,criterion="gini",random_state=SEED)
gini_tree.fit(gini_demo[["interaction_score"]],gini_demo["class"])

In [ ]:
plt.figure(figsize=(11,6))
plot_tree(gini_tree,feature_names=["interaction_score"],class_names=["Class 0","Class 1"],filled=True,rounded=True,impurity=True)
plt.title("Gini impurity split — classification concept only")
plt.show()

For the real DDG regression tree below, node impurity is regression error, not Gini.

In [ ]:
def around_int(v,low=1,step=2):
    return sorted(set([max(low,v-step),v,v+step]))

In [ ]:
def feature_family(name):
    if name.startswith("LIG."): return "LIG"
    if name.startswith("MUT."): return "MUT"
    if name.startswith("ENV."): return "ENV"
    if name.startswith("PLIP."): return "PLIP"
    if name=="vina" or name.startswith("vina"): return "VINA"
    if name.startswith(("P2.","P2dl.","P2dp.")): return "P2"
    if name.startswith("FE."): return "Engineered"
    return "Other"

# MODEL A — Decision Tree Regression

In [ ]:
XA=df_clean[A_FEATURES].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
XA.shape

In [ ]:
XA_dev=XA.iloc[dev_idx].reset_index(drop=True)
XA_test=XA.iloc[test_idx].reset_index(drop=True)

In [ ]:
pd.Series({"Development":len(XA_dev),"Test":len(XA_test),"Features":XA_dev.shape[1]})

In [ ]:
A_missing=XA_dev.isna().mean().sort_values(ascending=False)
A_missing.head(10).to_frame("Missing")

In [ ]:
tmp=A_missing.head(12).sort_values()
plt.figure(figsize=(7,4)); plt.barh(tmp.index,tmp.values)
plt.xlabel("Missing fraction"); plt.title("Model A missing values"); plt.tight_layout(); plt.show()

In [ ]:
A_constant=XA_dev.nunique(dropna=True)
A_constant=A_constant[A_constant<=1].index.tolist()
A_constant

In [ ]:
A_KEEP=[x for x in XA_dev.columns if x not in A_constant]
len(A_KEEP)

In [ ]:
XA_dev=XA_dev[A_KEEP]; XA_test=XA_test[A_KEEP]

In [ ]:
A_imputer=SimpleImputer(strategy="median")
XA_dev_imp=pd.DataFrame(A_imputer.fit_transform(XA_dev),columns=A_KEEP)
XA_test_imp=pd.DataFrame(A_imputer.transform(XA_test),columns=A_KEEP)

In [ ]:
A_unpruned=DecisionTreeRegressor(random_state=SEED)
A_unpruned.fit(XA_dev_imp,y_dev)

In [ ]:
pd.Series({"Depth":A_unpruned.tree_.max_depth,"Leaves":A_unpruned.tree_.n_leaves,"Nodes":A_unpruned.tree_.node_count})

In [ ]:
A_un_train=A_unpruned.predict(XA_dev_imp); A_un_test=A_unpruned.predict(XA_test_imp)
pd.DataFrame({"Split":["Development","Test"],"R2":[r2_score(y_dev,A_un_train),r2_score(y_test,A_un_test)]}).round(4)

In [ ]:
A_demo=DecisionTreeRegressor(max_depth=3,min_samples_leaf=20,random_state=SEED)
A_demo.fit(XA_dev_imp,y_dev)

In [ ]:
plt.figure(figsize=(22,10))
plot_tree(A_demo,feature_names=A_KEEP,filled=True,rounded=True,impurity=True,precision=3)
plt.title("Model A — regression split graph"); plt.show()

In [ ]:
sample_y=np.array([0.2,0.4,0.5,1.3,1.5])
parent_mse=np.mean((sample_y-sample_y.mean())**2)
parent_mse

In [ ]:
left_y=sample_y[:3]; right_y=sample_y[3:]
child_mse=(len(left_y)/len(sample_y))*np.mean((left_y-left_y.mean())**2)+(len(right_y)/len(sample_y))*np.mean((right_y-right_y.mean())**2)
parent_mse,child_mse,parent_mse-child_mse

In [ ]:
A_depth_values=[2,3,4,5,6,8,10,12,16,20,24,30,None]
rows=[]
for d in A_depth_values:
    m=DecisionTreeRegressor(max_depth=d,random_state=SEED).fit(XA_dev_imp,y_dev)
    rows.append({"depth":str(d),"Train_R2":m.score(XA_dev_imp,y_dev),"Test_R2":m.score(XA_test_imp,y_test)})
A_depth_df=pd.DataFrame(rows); A_depth_df

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(len(A_depth_df)),A_depth_df["Train_R2"],marker="o",label="Train")
plt.plot(range(len(A_depth_df)),A_depth_df["Test_R2"],marker="o",label="Test")
plt.xticks(range(len(A_depth_df)),A_depth_df["depth"],rotation=30)
plt.xlabel("max_depth"); plt.ylabel("R²"); plt.title("Model A depth vs overfitting"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
A_leaf_values=[1,2,3,5,8,12,20,30,50]
tr,va=validation_curve(DecisionTreeRegressor(max_depth=20,random_state=SEED),XA_dev_imp,y_dev,param_name="min_samples_leaf",param_range=A_leaf_values,cv=3,scoring="r2",n_jobs=N_JOBS)

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(A_leaf_values,tr.mean(1),marker="o",label="Train")
plt.plot(A_leaf_values,va.mean(1),marker="o",label="CV")
plt.xlabel("min_samples_leaf"); plt.ylabel("R²"); plt.title("Model A leaf-size curve"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
A_split_values=[2,5,10,20,40,80]
tr,va=validation_curve(DecisionTreeRegressor(max_depth=20,random_state=SEED),XA_dev_imp,y_dev,param_name="min_samples_split",param_range=A_split_values,cv=3,scoring="r2",n_jobs=N_JOBS)

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(A_split_values,tr.mean(1),marker="o",label="Train")
plt.plot(A_split_values,va.mean(1),marker="o",label="CV")
plt.xlabel("min_samples_split"); plt.ylabel("R²"); plt.title("Model A split-size curve"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
crit_rows=[]
for crit in ["squared_error","friedman_mse"]:
    m=DecisionTreeRegressor(criterion=crit,max_depth=16,min_samples_leaf=3,random_state=SEED).fit(XA_dev_imp,y_dev)
    crit_rows.append({"criterion":crit,**metrics(y_test,m.predict(XA_test_imp))})
pd.DataFrame(crit_rows).round(4)

In [ ]:
A_pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("model",DecisionTreeRegressor(random_state=SEED))])

In [ ]:
A_params={
"model__criterion":["squared_error","friedman_mse"],
"model__splitter":["best","random"],
"model__max_depth":[6,8,10,12,16,20,24,30,None],
"model__min_samples_split":[2,5,10,20,40],
"model__min_samples_leaf":[1,2,3,5,8,12,20],
"model__max_features":[None,"sqrt",0.5,0.75,1.0],
"model__max_leaf_nodes":[None,30,60,100,200,400]}

In [ ]:
A_search=RandomizedSearchCV(A_pipe,A_params,n_iter=40,scoring="neg_root_mean_squared_error",cv=5,random_state=SEED,n_jobs=N_JOBS)
A_search.fit(XA_dev,y_dev)

In [ ]:
A_search.best_params_

In [ ]:
-A_search.best_score_

In [ ]:
A_pre=A_search.best_estimator_
A_pre_pred=A_pre.predict(XA_test)
pd.Series(metrics(y_test,A_pre_pred)).round(4)

In [ ]:
A_core=A_pre.named_steps["model"]
A_path=A_core.cost_complexity_pruning_path(XA_dev_imp,y_dev)
len(A_path.ccp_alphas)

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(A_path.ccp_alphas[:-1],A_path.impurities[:-1])
plt.xlabel("ccp_alpha"); plt.ylabel("Leaf impurity"); plt.title("Model A pruning path"); plt.tight_layout(); plt.show()

In [ ]:
A_alphas=np.unique(np.quantile(A_path.ccp_alphas,np.linspace(.05,.95,18)))
A_alphas

In [ ]:
A_prune=[]
for alpha in A_alphas:
    p=A_search.best_params_
    m=DecisionTreeRegressor(
        criterion=p["model__criterion"],splitter=p["model__splitter"],
        max_depth=p["model__max_depth"],min_samples_split=p["model__min_samples_split"],
        min_samples_leaf=p["model__min_samples_leaf"],max_features=p["model__max_features"],
        max_leaf_nodes=p["model__max_leaf_nodes"],ccp_alpha=alpha,random_state=SEED)
    s=cross_validate(m,XA_dev_imp,y_dev,cv=3,scoring="neg_root_mean_squared_error",n_jobs=N_JOBS)
    A_prune.append({"ccp_alpha":alpha,"CV_RMSE":-s["test_score"].mean()})
A_prune_df=pd.DataFrame(A_prune); A_prune_df.head()

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(A_prune_df["ccp_alpha"],A_prune_df["CV_RMSE"],marker="o")
plt.xlabel("ccp_alpha"); plt.ylabel("CV RMSE"); plt.title("Model A pruning strength"); plt.tight_layout(); plt.show()

In [ ]:
A_best_alpha=A_prune_df.sort_values("CV_RMSE").iloc[0]["ccp_alpha"]
A_best_alpha

In [ ]:
p=A_search.best_params_
A_final=DecisionTreeRegressor(
criterion=p["model__criterion"],splitter=p["model__splitter"],max_depth=p["model__max_depth"],
min_samples_split=p["model__min_samples_split"],min_samples_leaf=p["model__min_samples_leaf"],
max_features=p["model__max_features"],max_leaf_nodes=p["model__max_leaf_nodes"],
ccp_alpha=A_best_alpha,random_state=SEED)
A_final.fit(XA_dev_imp,y_dev)

In [ ]:
A_test_pred=A_final.predict(XA_test_imp)
A_result=metrics(y_test,A_test_pred)
pd.Series(A_result).round(4)

In [ ]:
pd.Series({"Depth":A_final.tree_.max_depth,"Leaves":A_final.tree_.n_leaves,"Nodes":A_final.tree_.node_count})

In [ ]:
actual_vs_pred(y_test,A_test_pred,"Model A — final Decision Tree")

In [ ]:
A_residual=y_test.to_numpy()-A_test_pred
residual_plot(A_test_pred,A_residual,"Model A residuals")

In [ ]:
plt.figure(figsize=(6,4)); plt.hist(A_residual,bins=50)
plt.xlabel("Residual"); plt.ylabel("Frequency"); plt.title("Model A residual distribution"); plt.tight_layout(); plt.show()

In [ ]:
A_imp=pd.DataFrame({"Feature":A_KEEP,"Importance":A_final.feature_importances_}).sort_values("Importance",ascending=False)
A_imp.head(15).round(4)

In [ ]:
tmp=A_imp.head(15).sort_values("Importance")
plt.figure(figsize=(8,5)); plt.barh(tmp["Feature"],tmp["Importance"])
plt.xlabel("Importance"); plt.title("Model A feature importance"); plt.tight_layout(); plt.show()

In [ ]:
A_perm=permutation_importance(A_final,XA_test_imp,y_test,n_repeats=5,random_state=SEED,scoring="r2",n_jobs=N_JOBS)

In [ ]:
A_perm_df=pd.DataFrame({"Feature":A_KEEP,"Importance":A_perm.importances_mean}).sort_values("Importance",ascending=False)
A_perm_df.head(15).round(4)

In [ ]:
plt.figure(figsize=(22,10))
plot_tree(A_final,feature_names=A_KEEP,max_depth=3,filled=True,rounded=True,impurity=True,precision=3)
plt.title("Model A final tree — first 3 levels"); plt.show()

In [ ]:
A_sample=XA_test_imp.iloc[[0]]
A_leaf=A_final.apply(A_sample)[0]
A_leaf

In [ ]:
A_nodes=A_final.decision_path(A_sample).indices
A_nodes

In [ ]:
t=A_final.tree_; path=[]
for node in A_nodes:
    f=t.feature[node]
    if f>=0:
        val=A_sample.iloc[0,f]; thr=t.threshold[node]
        path.append({"Node":node,"Feature":A_KEEP[f],"Value":val,"Threshold":thr,"Direction":"Left" if val<=thr else "Right"})
pd.DataFrame(path).round(4)

In [ ]:
sizes,tr,va=learning_curve(A_final,XA_dev_imp,y_dev,cv=3,scoring="r2",train_sizes=np.linspace(.2,1,5),n_jobs=N_JOBS)

In [ ]:
plt.figure(figsize=(6,4)); plt.plot(sizes,tr.mean(1),marker="o",label="Train"); plt.plot(sizes,va.mean(1),marker="o",label="CV")
plt.xlabel("Training rows"); plt.ylabel("R²"); plt.title("Model A learning curve"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
A_error=pd.DataFrame({"Actual":y_test,"Predicted":A_test_pred,"Residual":A_residual})
A_error["AbsError"]=A_error["Residual"].abs()
A_error.sort_values("AbsError",ascending=False).head(12).round(4)

In [ ]:
pd.DataFrame({"SAMPLE_ID":df_clean.iloc[test_idx]["SAMPLE_ID"].values,"Actual":y_test,"Predicted":A_test_pred,"Residual":A_residual}).to_csv("DT_Model_A_predictions.csv",index=False)

In [ ]:
joblib.dump({"model":A_final,"imputer":A_imputer,"features":A_KEEP,"search":A_search.best_params_,"ccp_alpha":A_best_alpha},"DT_Model_A.joblib")

# MODEL B — PLIP-Aware Decision Tree

In [ ]:
XB=df_clean[B_FEATURES].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
XB.shape

In [ ]:
XB_dev=XB.iloc[dev_idx].reset_index(drop=True); XB_test=XB.iloc[test_idx].reset_index(drop=True)

In [ ]:
XB_dev[PLIP].describe().T[["mean","std","min","max"]].round(3)

In [ ]:
B_active=(XB_dev[PLIP]>0).mean().sort_values()
B_active.to_frame("Active fraction")

In [ ]:
plt.figure(figsize=(7,4)); plt.barh(B_active.index,B_active.values)
plt.xlabel("Active fraction"); plt.title("Model B PLIP activity"); plt.tight_layout(); plt.show()

In [ ]:
def add_B_features(X):
    X=X.copy()
    X["FE.PLIPTotal"]=X[PLIP].sum(axis=1,min_count=1)
    X["FE.PLIPMean"]=X[PLIP].mean(axis=1)
    X["FE.PLIPStd"]=X[PLIP].std(axis=1)
    X["FE.PLIPActive"]=(X[PLIP]>0).sum(axis=1)
    if "LIG.MolWt" in X:
        X["FE.PLIPPerMolWt"]=X["FE.PLIPTotal"]/X["LIG.MolWt"].replace(0,np.nan)
    return X.replace([np.inf,-np.inf],np.nan)

In [ ]:
XB_dev_fe=add_B_features(XB_dev); XB_test_fe=add_B_features(XB_test)
XB_dev_fe.shape

In [ ]:
B_constant=XB_dev_fe.nunique(dropna=True); B_constant=B_constant[B_constant<=1].index.tolist()
B_KEEP=[x for x in XB_dev_fe.columns if x not in B_constant]
len(B_KEEP)

In [ ]:
XB_dev_fe=XB_dev_fe[B_KEEP]; XB_test_fe=XB_test_fe[B_KEEP]
B_imputer=SimpleImputer(strategy="median")
XB_dev_imp=pd.DataFrame(B_imputer.fit_transform(XB_dev_fe),columns=B_KEEP)
XB_test_imp=pd.DataFrame(B_imputer.transform(XB_test_fe),columns=B_KEEP)

In [ ]:
B_probe=DecisionTreeRegressor(max_depth=8,min_samples_leaf=10,random_state=SEED).fit(XB_dev_imp,y_dev)

In [ ]:
B_probe_imp=pd.DataFrame({"Feature":B_KEEP,"Importance":B_probe.feature_importances_}).sort_values("Importance",ascending=False)
B_probe_imp.head(20).round(4)

In [ ]:
B_plip_imp=B_probe_imp[B_probe_imp["Feature"].str.startswith(("PLIP.","FE.PLIP"))]
B_plip_imp.round(4)

In [ ]:
plt.figure(figsize=(8,4)); plt.barh(B_plip_imp["Feature"],B_plip_imp["Importance"])
plt.xlabel("Importance"); plt.title("Model B PLIP importance"); plt.tight_layout(); plt.show()

In [ ]:
B_n=min(20000,len(XB_dev_imp))
B_idx=np.random.RandomState(SEED).choice(len(XB_dev_imp),B_n,replace=False)

In [ ]:
rows=[]
for crit in ["squared_error","friedman_mse","absolute_error"]:
    m=DecisionTreeRegressor(criterion=crit,max_depth=12,min_samples_leaf=5,random_state=SEED).fit(XB_dev_imp.iloc[B_idx],y_dev.iloc[B_idx])
    rows.append({"criterion":crit,**metrics(y_test,m.predict(XB_test_imp))})
B_criterion_df=pd.DataFrame(rows); B_criterion_df.round(4)

In [ ]:
plt.figure(figsize=(7,4)); plt.bar(B_criterion_df["criterion"],B_criterion_df["RMSE"])
plt.ylabel("RMSE"); plt.title("Model B criterion benchmark"); plt.tight_layout(); plt.show()

In [ ]:
B_params={
"criterion":["squared_error","friedman_mse"],"splitter":["best","random"],
"max_depth":[6,8,10,12,14,16,20,24,30],
"min_samples_split":[2,5,10,20,40,80],
"min_samples_leaf":[1,2,3,5,8,12,20,30],
"max_features":[None,"sqrt",0.4,0.6,0.8,1.0],
"max_leaf_nodes":[None,30,50,80,120,200,350]}

In [ ]:
B_search=RandomizedSearchCV(DecisionTreeRegressor(random_state=SEED),B_params,n_iter=55,scoring="neg_root_mean_squared_error",cv=5,random_state=SEED,n_jobs=N_JOBS)
B_search.fit(XB_dev_imp,y_dev)

In [ ]:
B_search.best_params_

In [ ]:
-B_search.best_score_

In [ ]:
bp=B_search.best_params_
B_depth_grid=sorted(set([max(3,bp["max_depth"]-2),bp["max_depth"],bp["max_depth"]+2]))

In [ ]:
B_refine_grid={
"criterion":[bp["criterion"]],"splitter":[bp["splitter"]],"max_depth":B_depth_grid,
"min_samples_split":around_int(bp["min_samples_split"],2,3),
"min_samples_leaf":around_int(bp["min_samples_leaf"],1,2),
"max_features":[bp["max_features"]],"max_leaf_nodes":[bp["max_leaf_nodes"]]}

In [ ]:
B_refine=GridSearchCV(DecisionTreeRegressor(random_state=SEED),B_refine_grid,scoring="neg_root_mean_squared_error",cv=5,n_jobs=N_JOBS)
B_refine.fit(XB_dev_imp,y_dev)

In [ ]:
B_refine.best_params_

In [ ]:
-B_refine.best_score_

In [ ]:
B_pre=B_refine.best_estimator_
B_path=B_pre.cost_complexity_pruning_path(XB_dev_imp,y_dev)

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(B_path.ccp_alphas[:-1],B_path.impurities[:-1])
plt.xlabel("ccp_alpha"); plt.ylabel("Leaf impurity"); plt.title("Model B pruning path"); plt.tight_layout(); plt.show()

In [ ]:
B_alphas=np.unique(np.quantile(B_path.ccp_alphas,np.linspace(.05,.95,20)))
B_alphas

In [ ]:
B_prune=[]
for alpha in B_alphas:
    m=DecisionTreeRegressor(**B_refine.best_params_,ccp_alpha=alpha,random_state=SEED)
    s=cross_validate(m,XB_dev_imp,y_dev,cv=3,scoring="neg_root_mean_squared_error",n_jobs=N_JOBS)
    B_prune.append({"ccp_alpha":alpha,"CV_RMSE":-s["test_score"].mean()})
B_prune_df=pd.DataFrame(B_prune); B_prune_df.head()

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(B_prune_df["ccp_alpha"],B_prune_df["CV_RMSE"],marker="o")
plt.xlabel("ccp_alpha"); plt.ylabel("CV RMSE"); plt.title("Model B pruning strength"); plt.tight_layout(); plt.show()

In [ ]:
B_best_alpha=B_prune_df.sort_values("CV_RMSE").iloc[0]["ccp_alpha"]
B_best_alpha

In [ ]:
B_final=DecisionTreeRegressor(**B_refine.best_params_,ccp_alpha=B_best_alpha,random_state=SEED).fit(XB_dev_imp,y_dev)

In [ ]:
B_test_pred=B_final.predict(XB_test_imp); B_result=metrics(y_test,B_test_pred)
pd.Series(B_result).round(4)

In [ ]:
pd.Series({"Depth":B_final.tree_.max_depth,"Leaves":B_final.tree_.n_leaves,"Nodes":B_final.tree_.node_count})

In [ ]:
actual_vs_pred(y_test,B_test_pred,"Model B — final Decision Tree")

In [ ]:
B_residual=y_test.to_numpy()-B_test_pred
residual_plot(B_test_pred,B_residual,"Model B residuals")

In [ ]:
B_imp=pd.DataFrame({"Feature":B_KEEP,"Importance":B_final.feature_importances_}).sort_values("Importance",ascending=False)
B_imp.head(20).round(4)

In [ ]:
tmp=B_imp.head(20).sort_values("Importance")
plt.figure(figsize=(8,6)); plt.barh(tmp["Feature"],tmp["Importance"])
plt.xlabel("Importance"); plt.title("Model B feature importance"); plt.tight_layout(); plt.show()

In [ ]:
B_perm=permutation_importance(B_final,XB_test_imp,y_test,n_repeats=5,random_state=SEED,scoring="r2",n_jobs=N_JOBS)

In [ ]:
B_perm_df=pd.DataFrame({"Feature":B_KEEP,"Importance":B_perm.importances_mean}).sort_values("Importance",ascending=False)
B_perm_df.head(20).round(4)

In [ ]:
plt.figure(figsize=(24,11))
plot_tree(B_final,feature_names=B_KEEP,max_depth=3,filled=True,rounded=True,impurity=True,precision=3)
plt.title("Model B final tree — first 3 levels"); plt.show()

In [ ]:
bt=B_final.tree_
B_nodes=pd.DataFrame({"Node":np.arange(bt.node_count),"Samples":bt.n_node_samples,"Impurity":bt.impurity})
B_nodes.head()

In [ ]:
plt.figure(figsize=(6,4)); plt.scatter(B_nodes["Samples"],B_nodes["Impurity"],s=10,alpha=.4)
plt.xlabel("Samples in node"); plt.ylabel("Node impurity"); plt.title("Model B node impurity"); plt.tight_layout(); plt.show()

In [ ]:
B_leaf_ids=B_final.apply(XB_test_imp); B_leaf_counts=pd.Series(B_leaf_ids).value_counts()
B_leaf_counts.describe().round(2)

In [ ]:
plt.figure(figsize=(6,4)); plt.hist(B_leaf_counts.values,bins=30)
plt.xlabel("Test samples per leaf"); plt.ylabel("Leaves"); plt.title("Model B leaf occupancy"); plt.tight_layout(); plt.show()

In [ ]:
B_top2=B_imp.head(2)["Feature"].tolist()
B_top2

In [ ]:
PartialDependenceDisplay.from_estimator(B_final,XB_dev_imp,features=B_top2)
plt.suptitle("Model B partial dependence"); plt.tight_layout(); plt.show()

In [ ]:
sizes,tr,va=learning_curve(B_final,XB_dev_imp,y_dev,cv=3,scoring="r2",train_sizes=np.linspace(.2,1,5),n_jobs=N_JOBS)

In [ ]:
plt.figure(figsize=(6,4)); plt.plot(sizes,tr.mean(1),marker="o",label="Train"); plt.plot(sizes,va.mean(1),marker="o",label="CV")
plt.xlabel("Training rows"); plt.ylabel("R²"); plt.title("Model B learning curve"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
AB_results=pd.DataFrame([{"Model":"A",**A_result},{"Model":"B",**B_result}])
AB_results.round(4)

In [ ]:
plt.figure(figsize=(5,4)); plt.bar(AB_results["Model"],AB_results["R2"])
plt.ylabel("R²"); plt.title("Decision Tree A vs B"); plt.tight_layout(); plt.show()

In [ ]:
pd.DataFrame({"SAMPLE_ID":df_clean.iloc[test_idx]["SAMPLE_ID"].values,"Actual":y_test,"Predicted":B_test_pred,"Residual":B_residual}).to_csv("DT_Model_B_predictions.csv",index=False)

In [ ]:
joblib.dump({"model":B_final,"imputer":B_imputer,"features":B_KEEP,"search":B_refine.best_params_,"ccp_alpha":B_best_alpha},"DT_Model_B.joblib")

# MODEL C — Full Decision Tree

In [ ]:
XC=df_clean[C_FEATURES].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
XC.shape

In [ ]:
XC_dev=XC.iloc[dev_idx].reset_index(drop=True); XC_test=XC.iloc[test_idx].reset_index(drop=True)

In [ ]:
C_family_missing=pd.Series({"LIG":XC_dev[LIG].isna().mean().mean(),"MUT":XC_dev[MUT].isna().mean().mean(),"ENV":XC_dev[ENV].isna().mean().mean(),"PLIP":XC_dev[PLIP].isna().mean().mean(),"VINA":XC_dev[VINA].isna().mean().mean(),"P2":XC_dev[P2].isna().mean().mean()})
C_family_missing.round(4)

In [ ]:
plt.figure(figsize=(7,4)); plt.bar(C_family_missing.index,C_family_missing.values)
plt.ylabel("Missing fraction"); plt.title("Model C missingness by family"); plt.tight_layout(); plt.show()

In [ ]:
def add_C_features(X):
    X=X.copy()
    X["FE.PLIPTotal"]=X[PLIP].sum(axis=1,min_count=1)
    X["FE.VinaMean"]=X[VINA].mean(axis=1)
    X["FE.VinaStd"]=X[VINA].std(axis=1)
    X["FE.VinaMin"]=X[VINA].min(axis=1)
    X["FE.VinaMax"]=X[VINA].max(axis=1)
    X["FE.VinaRange"]=X["FE.VinaMax"]-X["FE.VinaMin"]
    X["FE.VinaMedian"]=X[VINA].median(axis=1)
    p2=[x for x in P2 if x.startswith("P2.")]
    p2dl=[x for x in P2 if x.startswith("P2dl.")]
    p2dp=[x for x in P2 if x.startswith("P2dp.")]
    if p2: X["FE.P2Total"]=X[p2].sum(axis=1,min_count=1)
    if p2dl: X["FE.P2dlTotal"]=X[p2dl].sum(axis=1,min_count=1)
    if p2dp: X["FE.P2dpTotal"]=X[p2dp].sum(axis=1,min_count=1)
    if p2dl and p2dp: X["FE.P2Contrast"]=X["FE.P2dlTotal"]-X["FE.P2dpTotal"]
    return X.replace([np.inf,-np.inf],np.nan)

In [ ]:
XC_dev_fe=add_C_features(XC_dev); XC_test_fe=add_C_features(XC_test)
XC_dev_fe.shape

In [ ]:
C_constant=XC_dev_fe.nunique(dropna=True); C_constant=C_constant[C_constant<=1].index.tolist()
C_KEEP=[x for x in XC_dev_fe.columns if x not in C_constant]; len(C_KEEP)

In [ ]:
XC_dev_fe=XC_dev_fe[C_KEEP]; XC_test_fe=XC_test_fe[C_KEEP]
C_imputer=SimpleImputer(strategy="median")
XC_dev_imp=pd.DataFrame(C_imputer.fit_transform(XC_dev_fe),columns=C_KEEP)
XC_test_imp=pd.DataFrame(C_imputer.transform(XC_test_fe),columns=C_KEEP)

In [ ]:
C_probe=DecisionTreeRegressor(max_depth=12,min_samples_leaf=8,random_state=SEED).fit(XC_dev_imp,y_dev)

In [ ]:
C_probe_imp=pd.DataFrame({"Feature":C_KEEP,"Importance":C_probe.feature_importances_}).sort_values("Importance",ascending=False)
C_probe_imp.head(25).round(4)

In [ ]:
tmp=C_probe_imp.head(25).sort_values("Importance")
plt.figure(figsize=(9,7)); plt.barh(tmp["Feature"],tmp["Importance"])
plt.xlabel("Importance"); plt.title("Model C probe-tree importance"); plt.tight_layout(); plt.show()

In [ ]:
C_selector=SelectFromModel(C_probe,threshold="median",prefit=True)
C_SELECTED=np.array(C_KEEP)[C_selector.get_support()].tolist()
len(C_SELECTED)

In [ ]:
XC_dev_sel=XC_dev_imp[C_SELECTED]; XC_test_sel=XC_test_imp[C_SELECTED]

In [ ]:
C_selected_family=pd.Series([feature_family(x) for x in C_SELECTED]).value_counts()
C_selected_family

In [ ]:
plt.figure(figsize=(7,4)); plt.bar(C_selected_family.index,C_selected_family.values)
plt.ylabel("Selected features"); plt.title("Model C selected feature families"); plt.tight_layout(); plt.show()

In [ ]:
m1=DecisionTreeRegressor(max_depth=16,min_samples_leaf=5,random_state=SEED).fit(XC_dev_imp,y_dev)
m2=DecisionTreeRegressor(max_depth=16,min_samples_leaf=5,random_state=SEED).fit(XC_dev_sel,y_dev)
pd.DataFrame([{"Representation":"All",**metrics(y_test,m1.predict(XC_test_imp))},{"Representation":"Selected",**metrics(y_test,m2.predict(XC_test_sel))}]).round(4)

In [ ]:
C_params={
"criterion":["squared_error","friedman_mse"],"splitter":["best","random"],
"max_depth":[6,8,10,12,14,16,18,20,24,28,32],
"min_samples_split":[2,5,10,15,20,30,50,80],
"min_samples_leaf":[1,2,3,4,5,8,12,16,24,32],
"max_features":[None,"sqrt","log2",0.35,0.5,0.7,0.85,1.0],
"max_leaf_nodes":[None,30,50,80,120,180,250,400,600]}

In [ ]:
C_search=RandomizedSearchCV(DecisionTreeRegressor(random_state=SEED),C_params,n_iter=70,scoring="neg_root_mean_squared_error",cv=5,random_state=SEED,n_jobs=N_JOBS)
C_search.fit(XC_dev_sel,y_dev)

In [ ]:
C_search.best_params_

In [ ]:
-C_search.best_score_

In [ ]:
C_search_results=pd.DataFrame(C_search.cv_results_).sort_values("rank_test_score").head(15)
C_search_results[["rank_test_score","mean_test_score","param_max_depth","param_min_samples_leaf","param_min_samples_split","param_max_features","param_max_leaf_nodes"]]

In [ ]:
cp=C_search.best_params_
C_depth_grid=sorted(set([max(3,cp["max_depth"]-2),cp["max_depth"],cp["max_depth"]+2]))

In [ ]:
C_refine_grid={
"criterion":[cp["criterion"]],"splitter":[cp["splitter"]],"max_depth":C_depth_grid,
"min_samples_split":around_int(cp["min_samples_split"],2,3),
"min_samples_leaf":around_int(cp["min_samples_leaf"],1,2),
"max_features":[cp["max_features"]],"max_leaf_nodes":[cp["max_leaf_nodes"]]}

In [ ]:
C_refine=GridSearchCV(DecisionTreeRegressor(random_state=SEED),C_refine_grid,scoring="neg_root_mean_squared_error",cv=5,n_jobs=N_JOBS)
C_refine.fit(XC_dev_sel,y_dev)

In [ ]:
C_refine.best_params_

In [ ]:
-C_refine.best_score_

In [ ]:
C_pre=C_refine.best_estimator_
C_path=C_pre.cost_complexity_pruning_path(XC_dev_sel,y_dev)

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(C_path.ccp_alphas[:-1],C_path.impurities[:-1])
plt.xlabel("ccp_alpha"); plt.ylabel("Leaf impurity"); plt.title("Model C pruning path"); plt.tight_layout(); plt.show()

In [ ]:
C_alphas=np.unique(np.quantile(C_path.ccp_alphas,np.linspace(.03,.97,24)))
C_alphas

In [ ]:
C_prune=[]
for alpha in C_alphas:
    m=DecisionTreeRegressor(**C_refine.best_params_,ccp_alpha=alpha,random_state=SEED)
    s=cross_validate(m,XC_dev_sel,y_dev,cv=3,scoring="neg_root_mean_squared_error",n_jobs=N_JOBS)
    m.fit(XC_dev_sel,y_dev)
    C_prune.append({"ccp_alpha":alpha,"CV_RMSE":-s["test_score"].mean(),"Depth":m.tree_.max_depth,"Leaves":m.tree_.n_leaves})
C_prune_df=pd.DataFrame(C_prune); C_prune_df.head()

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(C_prune_df["ccp_alpha"],C_prune_df["CV_RMSE"],marker="o")
plt.xlabel("ccp_alpha"); plt.ylabel("CV RMSE"); plt.title("Model C pruning strength"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(C_prune_df["ccp_alpha"],C_prune_df["Leaves"],marker="o")
plt.xlabel("ccp_alpha"); plt.ylabel("Leaves"); plt.title("Model C alpha vs tree size"); plt.tight_layout(); plt.show()

In [ ]:
C_best_alpha=C_prune_df.sort_values("CV_RMSE").iloc[0]["ccp_alpha"]
C_best_alpha

In [ ]:
C_final=DecisionTreeRegressor(**C_refine.best_params_,ccp_alpha=C_best_alpha,random_state=SEED).fit(XC_dev_sel,y_dev)

In [ ]:
C_test_pred=C_final.predict(XC_test_sel); C_result=metrics(y_test,C_test_pred)
pd.Series(C_result).round(4)

In [ ]:
pd.Series({"Depth":C_final.tree_.max_depth,"Leaves":C_final.tree_.n_leaves,"Nodes":C_final.tree_.node_count})

In [ ]:
C_train_pred=C_final.predict(XC_dev_sel)
pd.DataFrame({"Split":["Development","Test"],"R2":[r2_score(y_dev,C_train_pred),r2_score(y_test,C_test_pred)],"RMSE":[np.sqrt(mean_squared_error(y_dev,C_train_pred)),np.sqrt(mean_squared_error(y_test,C_test_pred))]}).round(4)

In [ ]:
C_cv=cross_validate(C_final,XC_dev_sel,y_dev,cv=5,scoring={"R2":"r2","RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error"},n_jobs=N_JOBS)
pd.Series({"CV_R2":C_cv["test_R2"].mean(),"CV_RMSE":-C_cv["test_RMSE"].mean(),"CV_MAE":-C_cv["test_MAE"].mean()}).round(4)

In [ ]:
actual_vs_pred(y_test,C_test_pred,"Model C — final Decision Tree")

In [ ]:
C_residual=y_test.to_numpy()-C_test_pred
residual_plot(C_test_pred,C_residual,"Model C residuals")

In [ ]:
plt.figure(figsize=(6,4)); plt.hist(C_residual,bins=50)
plt.xlabel("Residual"); plt.ylabel("Frequency"); plt.title("Model C residual distribution"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(25,12))
plot_tree(C_final,feature_names=C_SELECTED,max_depth=3,filled=True,rounded=True,impurity=True,precision=3)
plt.title("Model C final tree — first 3 levels"); plt.show()

In [ ]:
C_imp=pd.DataFrame({"Feature":C_SELECTED,"Importance":C_final.feature_importances_}).sort_values("Importance",ascending=False)
C_imp.head(25).round(4)

In [ ]:
tmp=C_imp.head(25).sort_values("Importance")
plt.figure(figsize=(9,7)); plt.barh(tmp["Feature"],tmp["Importance"])
plt.xlabel("Importance"); plt.title("Model C feature importance"); plt.tight_layout(); plt.show()

In [ ]:
C_imp["Family"]=C_imp["Feature"].map(feature_family)
C_family_imp=C_imp.groupby("Family")["Importance"].sum().sort_values(ascending=False)
C_family_imp.round(4)

In [ ]:
plt.figure(figsize=(7,4)); plt.bar(C_family_imp.index,C_family_imp.values)
plt.ylabel("Total importance"); plt.title("Model C importance by family"); plt.tight_layout(); plt.show()

In [ ]:
C_perm=permutation_importance(C_final,XC_test_sel,y_test,n_repeats=7,random_state=SEED,scoring="r2",n_jobs=N_JOBS)

In [ ]:
C_perm_df=pd.DataFrame({"Feature":C_SELECTED,"Importance":C_perm.importances_mean,"Std":C_perm.importances_std}).sort_values("Importance",ascending=False)
C_perm_df.head(20).round(4)

In [ ]:
tmp=C_perm_df.head(20).sort_values("Importance")
plt.figure(figsize=(9,6)); plt.barh(tmp["Feature"],tmp["Importance"])
plt.xlabel("Permutation importance"); plt.title("Model C permutation importance"); plt.tight_layout(); plt.show()

In [ ]:
ct=C_final.tree_
depth=np.zeros(ct.node_count,dtype=int); stack=[(0,0)]
while stack:
    node,d=stack.pop(); depth[node]=d
    l=ct.children_left[node]; r=ct.children_right[node]
    if l!=r:
        stack.append((l,d+1)); stack.append((r,d+1))
C_nodes=pd.DataFrame({"Depth":depth,"Impurity":ct.impurity,"Samples":ct.n_node_samples})
C_nodes.head()

In [ ]:
C_depth_imp=C_nodes.groupby("Depth")["Impurity"].mean()
plt.figure(figsize=(7,4)); plt.plot(C_depth_imp.index,C_depth_imp.values,marker="o")
plt.xlabel("Tree depth"); plt.ylabel("Mean node impurity"); plt.title("Model C impurity by depth"); plt.tight_layout(); plt.show()

In [ ]:
C_leaf_ids=C_final.apply(XC_test_sel); C_leaf_counts=pd.Series(C_leaf_ids).value_counts()
C_leaf_counts.describe().round(2)

In [ ]:
plt.figure(figsize=(6,4)); plt.hist(C_leaf_counts.values,bins=30)
plt.xlabel("Test samples per leaf"); plt.ylabel("Leaves"); plt.title("Model C leaf occupancy"); plt.tight_layout(); plt.show()

In [ ]:
C_sample=XC_test_sel.iloc[[0]]; C_sample_leaf=C_final.apply(C_sample)[0]
C_sample_leaf

In [ ]:
C_sample_nodes=C_final.decision_path(C_sample).indices
C_sample_nodes

In [ ]:
path=[]
for node in C_sample_nodes:
    f=ct.feature[node]
    if f>=0:
        val=C_sample.iloc[0,f]; thr=ct.threshold[node]
        path.append({"Node":node,"Feature":C_SELECTED[f],"Value":val,"Threshold":thr,"Direction":"Left" if val<=thr else "Right"})
pd.DataFrame(path).round(4)

In [ ]:
C_top2=C_imp.head(2)["Feature"].tolist()
PartialDependenceDisplay.from_estimator(C_final,XC_dev_sel,features=C_top2)
plt.suptitle("Model C partial dependence"); plt.tight_layout(); plt.show()

In [ ]:
sizes,tr,va=learning_curve(C_final,XC_dev_sel,y_dev,cv=3,scoring="r2",train_sizes=np.linspace(.15,1,6),n_jobs=N_JOBS)

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(sizes,tr.mean(1),marker="o",label="Train"); plt.plot(sizes,va.mean(1),marker="o",label="CV")
plt.xlabel("Training rows"); plt.ylabel("R²"); plt.title("Model C learning curve"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
C_error=pd.DataFrame({"Actual":y_test,"Predicted":C_test_pred,"Residual":C_residual}); C_error["AbsError"]=C_error["Residual"].abs()
C_error.sort_values("AbsError",ascending=False).head(15).round(4)

In [ ]:
C_bins=pd.qcut(y_test,q=5,duplicates="drop")
C_bin=(pd.DataFrame({"Actual":y_test,"Predicted":C_test_pred,"Bin":C_bins}).groupby("Bin",observed=True).apply(lambda z:pd.Series({"N":len(z),"RMSE":np.sqrt(mean_squared_error(z["Actual"],z["Predicted"])),"MAE":mean_absolute_error(z["Actual"],z["Predicted"])})))
C_bin.round(4)

In [ ]:
plt.figure(figsize=(7,4)); plt.plot(range(len(C_bin)),C_bin["RMSE"],marker="o")
plt.xlabel("Target quintile"); plt.ylabel("RMSE"); plt.title("Model C error by target range"); plt.tight_layout(); plt.show()

In [ ]:
ABC_results=pd.DataFrame([{"Model":"A","Method":"Pre + post pruned DT",**A_result},{"Model":"B","Method":"PLIP-aware pruned DT",**B_result},{"Model":"C","Method":"Selected + pruned DT",**C_result}])
ABC_results.round(4)

In [ ]:
plt.figure(figsize=(6,4)); plt.bar(ABC_results["Model"],ABC_results["R2"])
plt.ylabel("R²"); plt.title("Decision Tree A vs B vs C"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6,4)); plt.bar(ABC_results["Model"],ABC_results["RMSE"])
plt.ylabel("RMSE"); plt.title("Decision Tree A/B/C RMSE"); plt.tight_layout(); plt.show()

In [ ]:
ABC_complexity=pd.DataFrame({"Model":["A","B","C"],"Depth":[A_final.tree_.max_depth,B_final.tree_.max_depth,C_final.tree_.max_depth],"Leaves":[A_final.tree_.n_leaves,B_final.tree_.n_leaves,C_final.tree_.n_leaves]})
ABC_complexity

In [ ]:
plt.figure(figsize=(6,4)); plt.bar(ABC_complexity["Model"],ABC_complexity["Leaves"])
plt.ylabel("Leaves"); plt.title("A/B/C final tree complexity"); plt.tight_layout(); plt.show()

In [ ]:
ABC_change=pd.DataFrame({"Transition":["A → B","B → C","A → C"],"Delta_R2":[B_result["R2"]-A_result["R2"],C_result["R2"]-B_result["R2"],C_result["R2"]-A_result["R2"]],"Delta_RMSE":[B_result["RMSE"]-A_result["RMSE"],C_result["RMSE"]-B_result["RMSE"],C_result["RMSE"]-A_result["RMSE"]]})
ABC_change.round(4)

In [ ]:
pd.DataFrame({"SAMPLE_ID":df_clean.iloc[test_idx]["SAMPLE_ID"].values,"Actual":y_test,"Predicted":C_test_pred,"Residual":C_residual}).to_csv("DT_Model_C_predictions.csv",index=False)

In [ ]:
ABC_results.to_csv("DT_Model_A_B_C_results.csv",index=False)

In [ ]:
joblib.dump({"model":C_final,"imputer":C_imputer,"selected_features":C_SELECTED,"search":C_refine.best_params_,"ccp_alpha":C_best_alpha},"DT_Model_C.joblib")

In [ ]:
final_table=ABC_results.copy()
final_table["Raw features"]=[len(A_FEATURES),len(B_FEATURES),len(C_FEATURES)]
final_table[["Model","Method","Raw features","R2","RMSE","MAE","Pearson","Spearman"]].round(4)

# Concepts covered
Gini concept demo, regression MSE splitting, recursive binary splits, criterion, splitter, max_depth, min_samples_split, min_samples_leaf, max_leaf_nodes, max_features, pre-pruning, cost-complexity post-pruning, ccp_alpha, RandomizedSearchCV, GridSearchCV refinement, overfitting curves, learning curves, feature importance, permutation importance, partial dependence, decision paths, node impurity, leaf occupancy, and A→B→C comparison.

## Export to the application
Run after all training cells. This verifies predictions against the locked test before exporting fitted preprocessing, models, feature lists, metrics, and plots.

In [ ]:
from backend.training.export import export_track
export_track(globals(), "decision_tree", PROJECT_ROOT)
